# 02 — Two-Proportion Z-Test

Test whether the difference in conversion rates between control and treatment is statistically significant.

---
**What this notebook covers:**
- Running a two-proportion z-test (two-sided and one-sided)
- Interpreting the z-score and p-value
- Understanding the effect of `alpha` on the decision
- Comparing the full `run_ab_test` pipeline result

In [ ]:
# ── path setup — run this cell first ─────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve().parent
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
from ab_testing_framework.validation import validate_input
from ab_testing_framework.z_test import perform_z_test

experiment = validate_input(10000, 450, 10000, 520)

# Two-sided test (default — most conservative, recommended)
result_two = perform_z_test(experiment, alternative='two-sided')
print(f'Two-sided  | z = {result_two.z_score:.4f}, p = {result_two.p_value:.4f}')

# One-sided test (use only when direction is pre-specified)
result_one = perform_z_test(experiment, alternative='larger')
print(f'One-sided  | z = {result_one.z_score:.4f}, p = {result_one.p_value:.4f}')

In [ ]:
# Decision at alpha = 0.05
alpha = 0.05
decision = 'Reject H₀' if result_two.p_value < alpha else 'Fail to reject H₀'
print(f'At α = {alpha}: {decision}')
print(f'Pooled rate: {result_two.pooled_rate:.4%}')

In [ ]:
# Full pipeline for comparison
from ab_testing_framework.analysis import run_ab_test

full = run_ab_test(10000, 450, 10000, 520, alpha=0.05, alternative='two-sided')
print(full.summary)
print()
print('Decision:       ', full.decision)
print('Recommendation: ', full.recommendation)

### Interpretation

- **z-score ≈ 3.36**: the observed difference is 3.36 standard errors above zero — well outside the null distribution.
- **p-value ≈ 0.0008** (two-sided): there is roughly a 0.08% chance of observing a difference this large if the null hypothesis were true.
- Since p < 0.05, we **reject H₀** and conclude the treatment has a statistically significant effect.
- The one-sided p-value is half the two-sided value — only use it if the direction was pre-specified *before* collecting data.

Proceed to `03_Confidence_Intervals.ipynb` to quantify the uncertainty around the lift.